In [ ]:
'''
Projet de Fin d'Études
Analyse Comportementale des Correcteurs et Optimisation des Troisièmes Corrections
Auteur : Telmoudy Mohamed Lemine
Date : 24.04.2025

Ce script réalise :
1. Génération d'un jeu de données synthétique de corrections BAC.
2. Feature engineering post-corrections : divergence locale, historique, divergence ajustée.
3. Préparation du train/test et modélisation (Logistic, RF, GB).
4. Documentation mathématique et statistique tout au long du pipeline.
'''

# 0. Imports et configuration ----------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# Pour reproductibilité
np.random.seed(42)

# 1. Définition du contexte et des paramètres ------------------------------------------------
# 1.1 Séries et matières (context pédagogique)
serie_courses = {
    'SN': ['Mathematiques', 'SciencesNaturelles', 'TarbietIslamia', 'Arabe', 'Francais', 'Anglais', 'Physique'],
    'LM': ['Philosophie', 'Mathematiques', 'TarbietIslamia', 'Arabe', 'Francais', 'Anglais', 'HistoireGeo'],
    'LO': ['FikrIslamique', 'CoranHadith', 'TarbietIslamia', 'Charia', 'Arabe', 'Mathematiques', 'HistoireGeo', 'Francais'],
    'M':  ['Mathematiques', 'SciencesNaturelles', 'TarbietIslamia', 'Arabe', 'Francais', 'Anglais', 'Physique']
}
# 1.2 Seuils dynamiques par matière (baseline fixe)  -----------------------------------------
# Ces seuils sont les marges acceptables initiales (en points) selon la difficulté perçue.
threshold_map = {
    'Mathematiques': 2.0, 'SciencesNaturelles': 2.0, 'Physique': 2.0,
    'Arabe': 2.5, 'Francais': 2.5, 'Anglais': 2.5,
    'TarbietIslamia': 3.0, 'Philosophie': 3.0,
    'FikrIslamique': 3.0, 'CoranHadith': 3.0, 'Charia': 3.0, 'HistoireGeo': 2.5
}
# 1.3 Création des correcteurs simulés --------------------------------------------------------
n_correctors = 20
correctors_by_matiere = {}
corrector_params = {}
for mat, thr in threshold_map.items():
    low_ids = [f"C_{mat[:3]}_L_{i+1}" for i in range(n_correctors//2)]
    high_ids = [f"C_{mat[:3]}_H_{i+1}" for i in range(n_correctors//2)]
    correctors_by_matiere[mat] = low_ids + high_ids
    # sigma low = thr/4, sigma high = thr
    sigma_low = thr / 4
    sigma_high = thr
    for cr in low_ids:
        corrector_params[cr] = {'bias': np.random.normal(0, sigma_low/3), 'sigma': sigma_low}
    for cr in high_ids:
        corrector_params[cr] = {'bias': np.random.normal(0, sigma_high/3), 'sigma': sigma_high}

# 2. Lecture des données officielles --------------------------------------------------------
bac = pd.read_csv('bac_results_2015-2024.csv')
# Garde uniquement les grades disponibles
bac = bac[bac['grade'].notna()].copy()
bac['grade'] = bac['grade'].astype(float)

# 3. Construction du dataset synthétique ----------------------------------------------------
records = []
id_copie = 0
for idx, row in bac.iterrows():
    serie = row['bac_serie'] if 'bac_serie' in row else row.get('serie', None)
    if serie not in serie_courses:
        continue
    year = row['year']
    note_A = row['grade']  # note réelle (invisible au modèle)
    for mat in serie_courses[serie]:
        id_copie += 1
        # Sélection aléatoire de deux correcteurs distincts
        pool = correctors_by_matiere[mat]
        crA, crB = np.random.choice(pool, 2, replace=False)
        # Simulation des notes post-corrections
        pA, pB = corrector_params[crA], corrector_params[crB]
        nA = np.random.normal(0, pA['sigma'])
        nB = np.random.normal(0, pB['sigma'])
        note_B_A = np.clip(note_A + pA['bias'] + nA, 0, 20)
        note_B_B = np.clip(note_A + pB['bias'] + nB, 0, 20)
        # Divergence locale = |note_B_X - note_B_other|
        diff_local = abs(note_B_A - note_B_B)
        # Décision initiale (baseline) sur 3e correction
        decl = int(np.mean([abs(note_A-note_B_A), abs(note_A-note_B_B)]) > threshold_map[mat])
        # Enregistrement
        records.append({
            'id_copie': id_copie, 'year': year, 'serie': serie, 'matiere': mat,
            'nni_A': crA, 'nni_B': crB,
            'note_A': note_A, 'note_B_A': note_B_A, 'note_B_B': note_B_B,
            'diff_local': diff_local, 'threshold': threshold_map[mat],
            'declenchement_3e': decl
        })

# DataFrame synthétique
df_syn = pd.DataFrame(records)


In [ ]:
df_syn.tail()

,id_copie,year,serie,matiere,nni_A,nni_B,note_A,note_B_A,note_B_B,diff_local,threshold,declenchement_3e,mean_diff_A,median_diff_A,mean_diff_B,median_diff_B,true_divergence_A,true_divergence_B
3345636,3345637,2024,LM,TarbietIslamia,C_Tar_L_5,C_Tar_L_2,5.979839,6.703313,5.449891,1.253421,3.0,0,1.680018,1.148490,1.682744,1.169969,0.426597,0.429323
3345637,3345638,2024,LM,Arabe,C_Ara_H_3,C_Ara_H_10,5.979839,8.283781,1.074369,7.209412,2.5,1,2.363588,1.945655,2.599983,2.190015,4.845824,4.609429
3345638,3345639,2024,LM,Francais,C_Fra_L_5,C_Fra_H_1,5.979839,7.071721,5.527800,1.543921,2.5,0,1.382906,0.951648,2.308766,1.902286,0.161015,0.764845
3345639,3345640,2024,LM,Anglais,C_Ang_H_1,C_Ang_H_2,5.979839,4.451663,7.539682,3.088018,2.5,0,2.621003,2.228018,2.378253,1.955643,0.467015,0.709765
3345640,3345641,2024,LM,HistoireGeo,C_His_L_6,C_His_L_7,5.979839,6.414248,5.588408,0.825840,2.5,0,1.447084,1.000173,1.412763,0.955570,0.621243,0.586923


In [ ]:

# 4. Feature engineering réaliste                   -----------------------------------------------------------
# 4.1 Statistiques globales de divergence par correcteur/matière
hist = df_syn[['nni_A','matiere','diff_local']].rename(columns={'nni_A':'correcteur'})
hist = pd.concat(
    [hist, df_syn[['nni_B','matiere','diff_local']].rename(columns={'nni_B':'correcteur'})],
    ignore_index=True
)
stats = hist.groupby(['correcteur','matiere'])['diff_local'] \
            .agg(mean_diff='mean', median_diff='median').reset_index()
# 4.2 Fusion des stats dans df_syn
for side in ['A','B']:
    df_syn = df_syn.merge(
        stats.rename(columns={
            'correcteur':f'nni_{side}',
            'mean_diff':f'mean_diff_{side}',
            'median_diff':f'median_diff_{side}'
        }), on=[f'nni_{side}','matiere'], how='left'
    )
# 4.3 Divergence ajustée (true divergence)
#    = diff_local - mean_diff_side
for side in ['A','B']:
    df_syn[f'true_divergence_{side}'] = (
        df_syn['diff_local'] - df_syn[f'mean_diff_{side}']
    ).abs()

# 5. Préparation du train/test --------------------------------------------------------------
# 5.1 Features candidates (post-corrections + historiques + contexte)
feature_cols = [
    'note_B_A', 'note_B_B',
    'diff_local',
    'true_divergence_A', 'true_divergence_B',
    'mean_diff_A', 'mean_diff_B',
    'matiere', 'serie',
    'threshold'
]
X = df_syn[feature_cols]
y = df_syn['declenchement_3e']

# 5.2 Split stratifié 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6. Pipeline de modélisation ----------------------------------------------------------------
# 6.1 Préprocesseur
num_feats = ['note_B_A','note_B_B','diff_local','true_divergence_A','true_divergence_B','mean_diff_A','mean_diff_B','threshold']
cat_feats = ['matiere','serie']
preproc = ColumnTransformer([
    ('num', StandardScaler(), num_feats),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_feats)
])

# 6.2 Définition des modèles
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# 6.3 Entraînement et évaluation
results = {}
for name, clf in models.items():
    pipe = Pipeline([('preproc', preproc), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_proba)
    print(f"{name} - ROC AUC: {auc:.3f}")
    print(classification_report(y_test, y_pred))
    results[name] = pipe

# 7. Sauvegarde des modèles ----------------------------------------------------------------
for name, pipe in results.items():
    joblib.dump(pipe, f'model_{name}.pkl')
    print(f"Modèle enregistré : model_{name}.pkl")


LogisticRegression - ROC AUC: 0.942
              precision    recall  f1-score   support

           0       0.98      0.85      0.91    593109
           1       0.41      0.85      0.55     76020

    accuracy                           0.85    669129
   macro avg       0.69      0.85      0.73    669129
weighted avg       0.91      0.85      0.87    669129

RandomForest - ROC AUC: 0.952
              precision    recall  f1-score   support

           0       0.95      1.00      0.97    593109
           1       0.94      0.61      0.74     76020

    accuracy                           0.95    669129
   macro avg       0.95      0.80      0.86    669129
weighted avg       0.95      0.95      0.95    669129

GradientBoosting - ROC AUC: 0.958
              precision    recall  f1-score   support

           0       0.95      1.00      0.97    593109
           1       0.99      0.59      0.74     76020

    accuracy                           0.95    669129
   macro avg       0.97     

In [ ]:
hist.tail ,stats.tail

NameError: name 'hist' is not defined